# K-Nearest Neighbors (KNN) Classifier

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## What is KNN?

Every algorithm so far had a training phase where weights got updated through gradient descent. KNN is fundamentally different — it **learns nothing during training**.

Instead, it uses a single powerful idea:

> *"To predict the outcome of a new match, find the K most similar matches in the training data and let them vote."*

### The algorithm in plain English

1. Store all training data (that's the entire "training" phase)
2. When a new match arrives, measure its **distance** to every training match in feature space
3. Find the **K nearest** training matches
4. **Majority vote** — predict whichever class appears most among those K neighbors

### Lazy vs Eager Learning

| | Perceptron / Lin. Reg. / Log. Reg. | KNN |
|---|---|---|
| Training | Heavy — updates weights over epochs | Trivial — just stores data |
| Prediction | Instant — just arithmetic | Slow — searches all training data |
| Memory | Only stores weights | Stores entire dataset |
| Parameters learned | Weights and bias | None — K is set by you |

KNN is called a **lazy learner** for exactly this reason — it defers all work to prediction time.

### What K controls

- **Small K** (e.g. K=1): very sensitive to individual points, can overfit
- **Large K**: averages over many neighbors, smoother but may miss local patterns
- K is a **hyperparameter** — set by you, not learned from data

### Why scaling matters even more here

KNN measures distance in feature space. If one feature has values in the hundreds and another has values near 0.5, the large feature will dominate the distance calculation regardless of how informative it actually is. StandardScaler is essential — not optional.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.knn import KNNClassifier

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Visualise the Concept — Decision Boundary in 2D

Before touching football data, let's see what KNN actually does geometrically on a simple toy dataset.

In [ ]:
# Toy dataset: two crescent-shaped classes (not linearly separable)
from sklearn.datasets import make_moons
X_toy, y_toy = make_moons(n_samples=200, noise=0.25, random_state=SEED)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, k in zip(axes, [1, 5, 50]):
    model_toy = KNNClassifier(k=k).fit(X_toy, y_toy)

    # Create a mesh of points to colour the decision regions
    xx, yy = np.meshgrid(
        np.linspace(X_toy[:, 0].min() - 0.5, X_toy[:, 0].max() + 0.5, 200),
        np.linspace(X_toy[:, 1].min() - 0.5, X_toy[:, 1].max() + 0.5, 200),
    )
    Z = model_toy.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=1)
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='RdBu',
               edgecolors='white', linewidth=0.5, s=25)
    ax.set_title(f'K = {k}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('KNN Decision Boundaries — Effect of K', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### What you're seeing

- **K=1**: The boundary is jagged and wraps tightly around every individual point. The model has memorized the training data — this is overfitting.
- **K=5**: Smoother boundary that still captures the crescent shape. Good balance.
- **K=50**: The boundary is very smooth but starts losing the local curvature — underfitting.

Notice something important: **KNN can learn non-linear boundaries** — something Logistic Regression cannot. Those curved decision regions would be impossible with a straight line.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate',
    'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Scaling is critical for KNN — distances must be computed on equal footing
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 4. Find the Best K — Hyperparameter Search

In [ ]:
k_values = list(range(1, 31))
train_scores, test_scores = [], []

for k in k_values:
    m = KNNClassifier(k=k).fit(X_train_sc, y_train)
    train_scores.append(m.score(X_train_sc, y_train))
    test_scores.append(m.score(X_test_sc, y_test))

best_k = k_values[np.argmax(test_scores)]
print(f'Best K on test set: {best_k} (accuracy: {max(test_scores):.3f})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(k_values, train_scores, label='Train accuracy', color='#4878CF', linewidth=2)
ax.plot(k_values, test_scores,  label='Test accuracy',  color='#E87272', linewidth=2)
ax.axvline(best_k, color='black', linestyle='--', linewidth=1.2, label=f'Best K = {best_k}')
ax.set_xlabel('K (number of neighbors)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy vs K — Bias-Variance Tradeoff', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Reading this plot

- **K=1**: Train accuracy = 100% (memorization), test accuracy is lower — classic overfitting
- As K increases: train accuracy drops, test accuracy rises then falls
- The best K is where test accuracy peaks — the sweet spot between overfitting and underfitting

This tension between fitting the training data well vs generalizing to new data is called the **bias-variance tradeoff** — one of the most fundamental concepts in all of machine learning.

---
## 5. Train Final Model with Best K

In [ ]:
model = KNNClassifier(k=best_k).fit(X_train_sc, y_train)

print(f'K = {best_k}')
print(f'Train accuracy: {model.score(X_train_sc, y_train):.3f}')
print(f'Test  accuracy: {model.score(X_test_sc,  y_test):.3f}')

---
## 6. Confusion Matrix & Classification Report

In [ ]:
y_pred = model.predict(X_test_sc)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=ax)
ax.set_title(f'Confusion Matrix — KNN (K={best_k})', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 7. Predicted Probabilities

In [ ]:
probs = model.predict_proba(X_test_sc)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(probs[y_test == 0], bins=20, alpha=0.6, color='#E87272',
        label='True: Draw/Away (0)', density=True)
ax.hist(probs[y_test == 1], bins=20, alpha=0.6, color='#4878CF',
        label='True: Home Win (1)', density=True)
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold 0.5')
ax.set_xlabel('Predicted Probability of Home Win', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('KNN Predicted Probability Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### KNN probabilities vs Logistic Regression probabilities

KNN's probabilities are discrete — they can only take values like 0/5, 1/5, 2/5, 3/5, 4/5, 5/5 (for K=5). That's why you see spikes at specific values rather than a smooth curve. Logistic Regression produces a continuous probability — any value between 0 and 1. This is one reason Logistic Regression is preferred when well-calibrated probabilities matter.

---
## 8. Euclidean vs Manhattan Distance

In [ ]:
for metric in ['euclidean', 'manhattan']:
    m = KNNClassifier(k=best_k, metric=metric).fit(X_train_sc, y_train)
    print(f'{metric:12s} — Test accuracy: {m.score(X_test_sc, y_test):.3f}')

---
## 9. Discussion — Does KNN Suit Football?

### What it does well
- **No assumptions about data shape** — unlike Logistic Regression, KNN doesn't assume a linear boundary. It can capture complex patterns.
- **Intuitive** — the "find similar past matches" idea maps naturally to how human analysts think.
- **No training time** — fit() is instant, which matters for frequently updated datasets.

### What it struggles with

1. **Slow prediction** — for every new match, we search through all 40,000+ training matches. This scales badly with dataset size.
2. **The curse of dimensionality** — in high-dimensional feature spaces, all points become roughly equidistant from each other. The concept of "nearest neighbor" breaks down. With 7 features we're borderline fine; with 100 features KNN would struggle.
3. **Sensitive to irrelevant features** — every feature contributes to distance equally. A noisy irrelevant feature hurts KNN more than it hurts Logistic Regression.
4. **No feature importance** — unlike weighted models, KNN gives you no insight into which features mattered.
5. **Memory hungry** — the entire training dataset must be kept in memory at all times.

### Comparison to previous algorithms

| | Log. Reg. | KNN |
|---|---|---|
| Decision boundary | Linear only | Non-linear ✓ |
| Probability output | Smooth, continuous | Discrete steps |
| Feature importance | Readable from weights | Not available |
| Prediction speed | Fast | Slow |
| Interpretability | High | Low |
| Football suitability | Moderate | Moderate |


---
## Summary

| | |
|---|---|
| **Algorithm type** | Instance-based (lazy) classifier |
| **Training** | Store data only — no learning |
| **Prediction** | Find K nearest neighbors, majority vote |
| **Key hyperparameter** | K — set by you, not learned |
| **Distance metrics** | Euclidean (straight line), Manhattan (city grid) |
| **Key concept introduced** | Bias-variance tradeoff, hyperparameter tuning, curse of dimensionality |
| **Football suitability** | Moderate — non-linear but slow and memory-heavy |
